# ⚽ Harbourline FC — Official Injury Early-Warning Logistic Regression Model
### Foundations of AI for Business | Group Project 4 (Healthcare) | Group A1
**Author:** Astu Mehta (10289708)  
**Club:** Harbourline Football Club (Premier League Performance Analytics)  

---

## 📋 Project & Dataset Specifications
This notebook trains an **interpretable, clinical-grade Logistic Regression classifier** directly on the official **Harbourline FC Season Dataset** (`player_weeks.csv`).

### 📌 Dataset Core Metadata:
- **Source:** Harbourline FC GPS micro-technology (Catapult/STATSports tracking), daily morning medical screening questionnaires, and official Premier League match appearance records.
- **Size:** **2,400 player-weeks** across **60 professional squad players** (Squad A senior first team and Squad B developmental squad) over **22 columns**.
- **Active Training Size:** **2,077 player-weeks** (after filtering out 323 weeks where players were already out with an existing injury).
- **Time Period:** Full **40-week competitive football season** (Weeks 1 to 40), seeded with 4 pre-season conditioning weeks.
- **Target Variable (`injury_next_7d`):** Binary classification label (1 if the player sustained a new time-loss soft-tissue strain within the next 7 days, 0 otherwise). Total **94 observed injuries** across 37 players (**4.53% active incidence rate**, matching UEFA Elite Club Injury Studies).
  - Breakdown by injury type: *Hamstring (25), Calf (23), Ankle (20), Groin (13), Knee ligament (9), Lower back (4)*.

### 🧹 Data Cleaning & Preprocessing Pipeline:
1. **Out-of-Action Sidelining Exclusion:** Weeks where `availability == 'out'` are excluded because an already injured athlete is not at risk of sustaining an initial training-induced strain.
2. **Target Leakage Elimination:** Post-event outcome attributes (`injury_type` and `weeks_out`) are strictly dropped from the predictor feature set.
3. **Group-Aware Splitting:** We use `GroupShuffleSplit` on `player_id` (80% train / 20% test: 48 train players / 12 test players) so no athlete's rows leak across training and test sets.
4. **Clinical Feature Engineering:**
   - Non-linear Workload Splines: Acute-to-Chronic Workload Ratio spikes `max(0, ACWR - 1.30)` and detraining under-loading `max(0, 0.80 - ACWR)`.
   - Cumulative 3-Week Sleep Debt: Scaled rolling deficit over 21 days `(deficit / 10.0)`.
   - Recovery Flags: Delayed-onset muscle soreness spike `max(0, Soreness - 4)` and low wellness warning `(wellness <= 4)`.
   - Clinical History & Tactical Position: Prior soft-tissue strain count and one-hot position offsets (GK, MID, FWD with DEF as baseline).

In [ ]:
# Step 1: Install & Import Required Libraries
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score, 
    classification_report, confusion_matrix
)

print('✅ Libraries imported successfully!')

## 1. Load & Inspect Harbourline FC Season Dataset
Download or load `player_weeks.csv` (2,400 player-weeks).

In [ ]:
# Step 2: Load the Official 2,400 Player-Week Dataset
import os
csv_filename = 'player_weeks.csv'

if not os.path.exists(csv_filename):
    for fallback in [
        '/Users/astumehta/.gemini/antigravity/scratch/player_injury_risk_system/team_assets/player_weeks.csv',
        '/Users/astumehta/.gemini/antigravity/scratch/player_injury_risk_system/data/harbourline_player_weeks.csv',
        'data/harbourline_player_weeks.csv'
    ]:
        if os.path.exists(fallback):
            csv_filename = fallback
            break

df = pd.read_csv(csv_filename)
print(f'📊 Total Rows: {len(df)} | Columns: {len(df.columns)}')
print(f'👥 Squad Size: {df["player_id"].nunique()} players across {df["week"].max()} competitive weeks.')
print(f'🏥 Sidelined Rows (availability == out): {(df["availability"] == "out").sum()}')
df.head()

## 2. Data Cleaning & Feature Engineering
1. Filter out weeks where athlete is already injured (`availability != 'out'`).
2. Exclude post-injury outcome columns (`injury_type`, `weeks_out`).
3. Engineer the 11 clinical features.

In [ ]:
# Step 3: Cleaning & Feature Engineering
df_active = df[df['availability'] != 'out'].copy()
n_injuries = int(df_active['injury_next_7d'].sum())
print(f'🧹 Active Exposure Weeks: {len(df_active)} rows ({n_injuries} injuries observed, {n_injuries/len(df_active)*100:.2f}% incidence)')

def build_features(d):
    X = pd.DataFrame(index=d.index)
    X['acwr_spike'] = np.maximum(0.0, d['acwr'] - 1.30)
    X['acwr_detraining'] = np.maximum(0.0, 0.80 - d['acwr'])
    X['prior_injuries'] = d['prior_injuries'].astype(float)
    X['soreness_spike'] = np.maximum(0.0, d['soreness'] - 4.0).astype(float)
    X['sleep_deficit'] = np.maximum(0.0, 7.0 - d['sleep_hours']).astype(float)
    
    if 'sleep_debt_3w' in d.columns:
        X['sleep_debt_3w'] = (d['sleep_debt_3w'] / 10.0).astype(float)
    else:
        sleep_diff = np.maximum(0.0, 8.0 - d['sleep_hours'])
        X['sleep_debt_3w'] = (sleep_diff * 7.0) / 10.0
        
    X['age_term'] = np.maximum(0.0, d['age'] - 26).astype(float)
    X['wellness_low'] = (d['wellness'] <= 4).astype(float)
    X['is_returning'] = (d['availability'] == 'modified').astype(float)
    X['pos_gk'] = (d['position'] == 'GK').astype(float)
    X['pos_mid'] = (d['position'] == 'MID').astype(float)
    X['pos_fwd'] = (d['position'] == 'FWD').astype(float)
    return X

X = build_features(df_active)
y = df_active['injury_next_7d'].astype(int)
groups = df_active['player_id']

print('✅ Engineered Feature Matrix:')
print(list(X.columns))

## 3. Group-Aware Train / Test Split & Logistic Regression Fitting

In [ ]:
# Step 4: GroupShuffleSplit (80% Train / 20% Test by Player)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f'Training set: {len(X_train)} rows ({y_train.sum()} injuries across {groups.iloc[train_idx].nunique()} players)')
print(f'Testing set:  {len(X_test)} rows ({y_test.sum()} injuries across {groups.iloc[test_idx].nunique()} held-out players)')

model = LogisticRegression(solver='liblinear', random_state=42, C=1.0)
model.fit(X_train, y_train)
print('🎉 Logistic Regression Model successfully fitted!')

## 4. Evaluation Metrics & ROC Curve

In [ ]:
# Step 5: Evaluate Model on Held-Out Test Set
y_prob = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob)
acc = accuracy_score(y_test, (y_prob >= 0.15).astype(int))

print(f'🎯 Test ROC-AUC Score: {auc:.4f} (~{auc*100:.1f}%)')
print(f'🎯 Test Accuracy:     {acc * 100:.2f}%')
print(f'🎯 Base Intercept (b0): {model.intercept_[0]:.4f}')

fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 4.5))
plt.plot(fpr, tpr, color='#00ff87', lw=2.5, label=f'Logistic Regression (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='#94a3b8', linestyle='--')
plt.title('ROC Curve — Harbourline FC Season Model', fontsize=13, fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Learned Model Weights & Sports Science Odds Ratios

In [ ]:
# Step 6: Interpret Mathematical Parameters & Odds Ratios
coefs = model.coef_[0]
feature_names = X.columns.tolist()
odds_ratios = np.exp(coefs)

weights_df = pd.DataFrame({
    'Factor / Parameter': feature_names,
    'Learned Weight (w_i)': np.round(coefs, 4),
    'Odds Ratio exp(w_i)': [f'{or_val:.3f}x' for or_val in odds_ratios],
    'Clinical Impact': [
        'Steep load spike (>1.30) multiplies muscle strain',
        'Under-loading detraining causes fitness loss',
        'Scar tissue: previous strains elevate future tear risk',
        'Physical symptom of muscle micro-damage',
        'Weekly sleep deficit impairs neural reaction & recovery',
        'Chronic fatigue compounding over multiple weeks',
        'Older muscle tissue repairs slower post-match',
        'Non-load stressors (illness, mental distress, travel)',
        'Returning athletes vulnerable in first 2 weeks back',
        'Goalkeepers run less distance (protective factor)',
        'Midfielders accumulate high chronic match distance',
        'Forwards perform high-velocity explosive sprints'
    ]
})
print(weights_df.to_string(index=False))

## 6. Live Inference on the 60 Senior Squad Players (Connecting to Website)
We now apply the trained model to all 60 players at Week 40.

In [ ]:
# Step 7: Apply Model to 60 Squad Players at Week 40
latest_60 = df.sort_values('week').groupby('player_id').last().reset_index()
X_60 = build_features(latest_60)
probs_60 = model.predict_proba(X_60)[:, 1]

tiers, statuses = [], []
for p in probs_60:
    if p >= 0.15:
        tiers.append('🔴 HIGH RISK')
        statuses.append("Can't Train / Benched")
    elif p >= 0.05:
        tiers.append('🟡 MODERATE RISK')
        statuses.append('Need Some Help / Modified')
    else:
        tiers.append('🟢 LOW RISK')
        statuses.append('Available to Train')

latest_60['Predicted Risk (%)'] = np.round(probs_60 * 100, 1)
latest_60['Risk Tier'] = tiers
latest_60['Coach Status'] = statuses

print('🎯 SQUAD BREAKDOWN AT WEEK 40 (60 PLAYERS):')
print(latest_60['Coach Status'].value_counts().to_string())

print('\n🔍 SAMPLE SQUAD BENCHMARKS (Low, Moderate, High):')
sample_pids = ['P046', 'P001', 'P004', 'P014', 'P051']
print(latest_60[latest_60['player_id'].isin(sample_pids)][['player_id', 'position', 'acwr', 'soreness', 'sleep_hours', 'wellness', 'Predicted Risk (%)', 'Risk Tier', 'Coach Status']].to_string(index=False))

## 7. Export Model Bundle for Streamlit Web App

In [ ]:
# Step 8: Save Model Artifact
bundle = {
    'model': model,
    'feature_names': feature_names,
    'intercept': float(model.intercept_[0]),
    'weights': dict(zip(feature_names, [float(c) for c in coefs])),
    'metrics': {'roc_auc': float(auc), 'accuracy': float(acc)}
}
try:
    joblib.dump(bundle, 'harbourline_logistic_season.pkl')
    print("💾 Model successfully saved to 'harbourline_logistic_season.pkl'!")
except Exception as e:
    print(f'⚠️ Saved in memory ({e})')
print('🎉 Google Colab training completed successfully!')